In [ ]:
%load_ext autoreload
%autoreload 2
import ast, re
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_validate
from src.config import *
from src.utils import *
from src.experiments import run_experiment, show, augmented_cv, build_models, CV, MULTI_SCORING

base = pd.read_csv(MULTI_FEATURES_TRAIN_CSV)
base_aug = pd.read_csv(MULTI_FEATURES_AUG_CSV)
train = base.merge(pd.read_csv(MULTI_REGIONAL_TRAIN_CSV).drop(columns="label"), on="filename")
aug = base_aug.merge(pd.read_csv(MULTI_REGIONAL_AUG_CSV).drop(columns="label"), on="filename")
assert (train["filename"] == base["filename"]).all() and (aug["filename"] == base_aug["filename"]).all()

base_cols = [c for c in base.columns if c not in ("filename", "label")]
f1_cols = base_cols + [c for c in train.columns if c.startswith("reg_")]

y = train["label"].values
y_all = np.concatenate([y, aug["label"].values])
splits = augmented_cv(train["filename"], aug["source_filename"], y_all)   # same test folds as CV

def data_for(cols, augmented):
    """X, y and folds for one feature set, with or without the augmented scc copies."""
    if augmented:
        return np.vstack([train[cols].values, aug[cols].values]), y_all, splits
    return train[cols].values, y, CV

print(len(base_cols), "base |", len(f1_cols), "base + regional |", len(train), "originals |", len(aug), "augmented")

In [ ]:
from src.feature_extraction import draw_regions

s = random_sample(train)
show_grid([draw_regions(load_preprocessed(r.filename, r.label, MULTI_PREPROC_TRAIN_DIR, MULTI_CLASS_FOLDERS))
           for r in s.itertuples()], [f"{r.filename} | {MULTI_CLASS_NAMES[r.label]}" for r in s.itertuples()])

In [ ]:
# new name: (old name, augmented, weighted)
EXPERIMENTS = {
    "MF0 no handling":            ("M0 no handling", False, False),
    "MF1 class weights":          ("M1 class weights", False, True),
    "MF2 augmentation":           ("M2 augmentation", True, False),
    "MF3 augmentation + weights": ("M3 augmentation + weights", True, True),
}

tables, fitted = [], {}
for name, (_, augmented, weighted) in EXPERIMENTS.items():
    X_, y_, cv_ = data_for(f1_cols, augmented)
    table, fit = run_experiment(name, X_, y_, task="multiclass", weighted=weighted, cv=cv_)
    tables.append(table)
    fitted.update({(name, m): f for m, f in fit.items()})
    pd.concat(tables, ignore_index=True).to_csv(RESULTS_DIR / "multi_exp_regional.csv", index=False)

new = pd.concat(tables, ignore_index=True)
show(new.sort_values("kappa", ascending=False), task="multiclass")

In [ ]:
def parse(p):
    """Params from the CSV (text with np.float64(...)) or already a dict."""
    return ast.literal_eval(re.sub(r"np\.float64\(([^)]*)\)", r"\1", p)) if isinstance(p, str) else p

old = pd.concat([pd.read_csv(RESULTS_DIR / "multi_exp_baseline.csv"),
                 pd.read_csv(RESULTS_DIR / "multi_exp_augmentation.csv")], ignore_index=True)

folds = []
for new_name, (old_name, augmented, weighted) in EXPERIMENTS.items():
    for name, table, cols in [(old_name, old, base_cols), (new_name, new, f1_cols)]:
        best = table[table["experiment"] == name].nlargest(1, "kappa").iloc[0]
        model = build_models(weighted=weighted)[best["model"]].set_params(**parse(best["params"]))
        X_, y_, cv_ = data_for(cols, augmented)
        s = cross_validate(model, X_, y_, cv=cv_, scoring=MULTI_SCORING, n_jobs=-2)
        folds.append(pd.DataFrame({m: s[f"test_{m}"] for m in MULTI_SCORING})
                     .assign(experiment=name, model=best["model"], fold=range(5)))
folds = pd.concat(folds, ignore_index=True)
folds.to_csv(RESULTS_DIR / "multi_exp_regional_folds.csv", index=False)

k = folds.pivot(index="fold", columns="experiment", values="kappa")
scc = folds.pivot(index="fold", columns="experiment", values="recall_scc")
rows = []
for new_name, (old_name, *_) in EXPERIMENTS.items():
    d = k[new_name] - k[old_name]
    rows.append({"experiment": new_name, "kappa base": k[old_name].mean(), "kappa + regional": k[new_name].mean(),
                 "gain": d.mean(), "std": d.std(), "folds won": (d > 0).sum(),
                 "scc recall base": scc[old_name].mean(), "scc recall + regional": scc[new_name].mean()})
pd.DataFrame(rows).round(3)